# Sanskrit Meter Identification 

**June 23, 2026**

This notebook uses the `chanda` [PyPI library](https://pypi.org/project/chanda/), which is part of Chandojñānam.

**Steps**: analyse one line, inspect its syllable weights, analyse a four-line verse, and compare exact with fuzzy matching

## 1.Input and Output

Input: is a verse line or a set of verse lines
- The analyser segments the text, marks syllables as laghu/guru, computes a gaṇa signature
- Looks for definitions compatible with that signature 

In [207]:
try:
    import chanda
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "chanda>=1.1,<2", "indic-transliteration"] )

In [208]:
from importlib.metadata import version
from chanda import Chanda
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

analyser = Chanda()
print("chanda", version("chanda"))
print("meter definitions loaded:", len(analyser.CHANDA))

chanda 1.1.0
meter definitions loaded: 630


## 2. Analyse one line
`analyze_text` handles script detection. The result below is one half-verse, so the analyser can report compatible anuṣṭubh pāda positions.

In [209]:
text = "उक्ता वसन्ततिलका तभजा जगौ गः"
analysis = analyser.analyze_text(text)
line = analysis.result.line[0].result

print("detected scheme:", line.scheme)
print("metrical length:", line.length)
print("syllable units:  ", line.syllables)
print("laghu/guru:      ", line.lg)
print("gaṇa signature:  ", line.gana)
print("matches:         ", line.chanda)

detected scheme: devanagari
metrical length: 14
syllable units:   ['उ', 'क्ता', 'व', 'स', 'न्त', 'ति', 'ल', 'का', 'त', 'भ', 'जा', 'ज', 'गौ', 'गः']
laghu/guru:       ['ग', 'ग', 'ल', 'ग', 'ल', 'ल', 'ल', 'ग', 'ल', 'ल', 'ग', 'ल', 'ग', 'ग']
gaṇa signature:   तभजजगग
matches:          [('वसन्ततिलका = सिंहोन्नता = सिंहोद्धता = उद्धर्षिणी', ('',))]


### Reading the Output

The lists `syllables` and `lg` are aligned. Some consonant-only units receive an empty weight and do not increase `length`. Use `line.length`, rather than `len(line.syllables)`, for the metrical count.

In [210]:
for syllable, weight in zip(line.syllables, line.lg):
    if weight:
        print(f"{syllable:>5} : {weight}")

    उ : ग
 क्ता : ग
    व : ल
    स : ग
  न्त : ल
   ति : ल
    ल : ल
   का : ग
    त : ल
    भ : ल
   जा : ग
    ज : ल
   गौ : ग
   गः : ग


## 3. The same text in ISO
Scheme names accepted by `chanda` are lower-case strings such as `iast`, which correspond to the names of the constants used by `indic_transliteration`.

In [211]:
text_iso = transliterate(text, sanscript.DEVANAGARI, sanscript.ISO)
iso_analysis = analyser.analyze_text(text_iso, scheme=sanscript.ISO)
iso_line = iso_analysis.result.line[0].result

print(text_iso)
print(iso_line.gana, iso_line.chanda)

uktā vasantatilakā tabhajā jagau gaḥ
तभजजगग [('वसन्ततिलका = सिंहोन्नता = सिंहोद्धता = उद्धर्षिणी', ('',))]


## 4. Analyse a complete verse
With `verse=True`, the analyser combines evidence across four lines instead of treating the lines only as independent inputs.

In [212]:
shalini_example = """माता रामो मत्पिता रामचन्द्रः
स्वामी रामो मत्सखा रामचन्द्रः
सर्वस्वं मे रामचन्द्रो दयालुर्
नान्यं जाने नैव जाने न जाने"""

verse_analysis = analyser.analyze_text(shalini_example, verse=True)
for item in verse_analysis.result.line:
    result = item.result
    print(item.index + 1, result.gana, result.chanda)

print("Chanda:", verse_analysis.result.verse[0].chanda)

1 मततगग [('शालिनी', ('',))]
2 मततगग [('शालिनी', ('',))]
3 मततगग [('शालिनी', ('',))]
4 मततगग [('शालिनी', ('',))]
Chanda: (['शालिनी'], 4.0)


## 5. Exact and fuzzy matching
Fuzzy matching proposes nearby metrical patterns when exact matching fails. Treat the suggestion as a diagnostic: it may indicate a typo, a textual variant, a licence, or simply the wrong expected meter.

In [213]:
expected = "कश्चित्कान्ताविरहगुरुणा स्वाधिकारात्प्रमत्तः"
altered =  "कश्चित्कान्ताविरहगुरुणा स्वधिकारप्रमत्तः"

exact = analyser.analyze_line(expected)
without_fuzzy = analyser.analyze_line(altered)
with_fuzzy = analyser.analyze_line(altered, fuzzy=True)

print("expected:", exact.found, exact.chanda)
print("altered, exact only:", without_fuzzy.found)
print("best fuzzy suggestion:", with_fuzzy.fuzzy[0]["chanda"])
print("edit cost" , with_fuzzy.fuzzy[0]["cost"])
print("similarity:", with_fuzzy.fuzzy[0]["similarity"])

expected: True [('मन्दाक्रान्ता', ('',))]
altered, exact only: False
best fuzzy suggestion: [('मन्दाक्रान्ता', ('',))]
edit cost 1
similarity: 0.9411764705882353


## 6. Inspect bundled examples
The package ships with demonstration inputs for six meters. Compare each expected label with the verse-level decision; aliases or mismatches are useful regression cases to investigate.

In [214]:
examples = analyser.read_examples()
for meter, lines in examples.items():
    example_text = "\n".join(lines[2:])
    result = analyser.analyze_text(example_text, verse=True)
    decisions = result.result.verse[0].chanda[0]
    matched = any(meter in name for name in decisions)
    status = "match" if matched else "review"
    print(f"{status:6} expected={meter:18} first decision={decisions[0]}")

match  expected=शालिनी             first decision=शालिनी
match  expected=इन्द्रवज्रा        first decision=इन्द्रवज्रा
match  expected=वसन्ततिलका         first decision=वसन्ततिलका = सिंहोन्नता = सिंहोद्धता = उद्धर्षिणी
review expected=भुजङ्गप्रयात       first decision=सोमराजी
review expected=पञ्चचामर           first decision=लासिनी
match  expected=शार्दूलविक्रीडित   first decision=शार्दूलविक्रीडित


## Writing a Simple Meter Identifier

We will now write a simple identifier for three meters: भुजङ्गप्रयात (bhujaṅgaprayāta), शार्दूलविक्रीडितम् (śardūlavikrīḍitam) and पृथ्वी (pr̥thvī)

* भुजङ्गप्रयात = यययय
* शार्दूलविक्रीडित = मसजसततग
* पृथ्वी = जसजसयलग

**Disclaimer**: This will not cover all the cases. 

### Metrical Rules

* Syllables with short (laghu) vowels are marked as guru
* Syllables with long (guru) vowels are marked as guru
* Syllables with short (laghu) vowels that precede joint letters are also marked as guru
* Last syllable of a line is marked as guru

In [215]:
try:
    import sanskrit_text as skt
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sanskrit_text"] )

In [216]:
def is_joint(syllable):
    return (skt.HALANTA in syllable) and (len(syllable) > 2)

In [217]:
print(is_joint("क्ष"))
print(skt.is_laghu("कि"))

True
True


In [218]:
DEFINITIONS = {
    "LGGLGGLGGLGG": "भुजङ्गप्रयात",
    "GGGLLGLGLLLGGGLGGLG": "शार्दूलविक्रीडित",
    "LGLLLGLGLLLGLGGLG": "पृथ्वी"
}

In [219]:
def get_metrical_signature(text_line):
    clean_line = skt.clean(text_line)
    syllables = skt.get_syllables_word(clean_line)
    syllable_pairs = list(zip(syllables, syllables[1:] + [""]))
    lg = []

    for syllable, next_syllable in syllable_pairs:
        if is_joint(next_syllable) or not skt.is_laghu(syllable):
            lg.append("G")
        else:
            lg.append("L")

    signature = "".join(lg)
    return signature


def identify_meter_from_line(text_line):
    signature = get_metrical_signature(text_line)
    meter_name = DEFINITIONS.get(signature, None)
    return signature, meter_name

In [220]:
lines = [
    "भुजङ्गमपि कोपितं शिरसि पुष्पवद् धारयेत्",
    "न तु प्रतिनिविष्टमूर्खजनचित्तमाराधयेत्",
    "विद्या नाम नरस्य रूपमधिकं प्रच्छन्नगुप्तं धनम्",
    "विद्या भोगकरी यशः सुखकरी विद्या गुरूणां गुरुः",
    "विद्या बन्धुजनो विदेशगमने विद्या परा देवता",
    "विद्या राजसु पूज्यते न हि धनं विद्याविहीनः पशुः",
    "महामङ्गले पुण्यभूमे त्वदर्थे",
    "पतत्वेष कायो नमस्ते नमस्ते",
    "योऽन्तः प्रविश्य मम वाचमिमां प्रसुप्तां",
    "सञ्जीवयत्यखिलशक्तिधरः स्वधाम्ना",
    "अन्यांश्च हस्तचरणश्रवणत्वगादीन्",
    "प्राणान्नमो भगवते पुरुषाय तुभ्यम्"
]

for line in lines:
    print(line)
    signature, meter_name = identify_meter_from_line(line)
    print("metrical signature:", signature)
    print("identified meter:", meter_name)
    print()

भुजङ्गमपि कोपितं शिरसि पुष्पवद् धारयेत्
metrical signature: LGLLLGLGLLLGLGGLGL
identified meter: None

न तु प्रतिनिविष्टमूर्खजनचित्तमाराधयेत्
metrical signature: LGLLLGLGLLLGLGGLGL
identified meter: None

विद्या नाम नरस्य रूपमधिकं प्रच्छन्नगुप्तं धनम्
metrical signature: GGGLLGLGLLLGGGLGGLLL
identified meter: None

विद्या भोगकरी यशः सुखकरी विद्या गुरूणां गुरुः
metrical signature: GGGLLGLGLLLGGGLGGLG
identified meter: शार्दूलविक्रीडित

विद्या बन्धुजनो विदेशगमने विद्या परा देवता
metrical signature: GGGLLGLGLLLGGGLGGLG
identified meter: शार्दूलविक्रीडित

विद्या राजसु पूज्यते न हि धनं विद्याविहीनः पशुः
metrical signature: GGGLLGLGLLLGGGLGGLG
identified meter: शार्दूलविक्रीडित

महामङ्गले पुण्यभूमे त्वदर्थे
metrical signature: LGGLGGLGGLGG
identified meter: भुजङ्गप्रयात

पतत्वेष कायो नमस्ते नमस्ते
metrical signature: LGGLGGLGGLGG
identified meter: भुजङ्गप्रयात

योऽन्तः प्रविश्य मम वाचमिमां प्रसुप्तां
metrical signature: GGGLGLLLGLLGLGG
identified meter: None

सञ्जीवयत्यखिलशक्तिधरः स्वधाम्ना


## Summary

- Meter identification is a sequence of explicit transformations: text → syllable units → weights → signature → candidate definition search
- Fuzzy output is a hypothesis for inspection, not automatic textual correction.

Resources: [web interface](https://sanskrit.iitk.ac.in/jnanasangraha/chanda/) · [source code](https://github.com/hrishikeshrt/chanda) · [published paper (WSC 2023)](https://hrishikeshrt.github.io/publication/wsc2023_2/)